In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "ic" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"


In [1]:
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from tqdm import tqdm

In [2]:
MODEL_NAME = "melll-uff/bertweetbr"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at melll-uff/bertweetbr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64005, 768, padding_idx=1)
    (position_embeddings): Embedding(130, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [3]:
import os
from pathlib import Path

def get_embeddings_batch(texts, batch_size=32):
    model.eval()
    all_embeds = []

    with torch.inference_mode():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i + batch_size]

            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)
            batch_embeddings = batch_embeddings.cpu().numpy().astype("float32")

            all_embeds.append(batch_embeddings)

            del inputs, outputs, batch_embeddings

    return np.vstack(all_embeds)

def get_embeddings_to_disk(
    texts,
    save_path,
    batch_size=16
):
    os.makedirs(save_path, exist_ok=True)

    model.eval()
    idx = 0

    with torch.inference_mode():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i + batch_size]

            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)

            batch_embeddings = batch_embeddings.cpu().numpy().astype("float32")

            np.save(
                f"{save_path}/embeddings_{idx}.npy",
                batch_embeddings
            )

            idx += 1

            del inputs, outputs, batch_embeddings


In [4]:
df_train = pd.read_parquet(
    DATA_DIR / "tweets_rotulados.parquet",
    
)

df_train = df_train.dropna(subset=["clean_text", "is_economic"])

texts_train = df_train["clean_text"].tolist()
y = df_train["is_economic"].values

In [5]:
X = get_embeddings_batch(texts_train, batch_size=32)

print("Shape dos embeddings:", X.shape)


100%|██████████| 13/13 [00:36<00:00,  2.77s/it]

Shape dos embeddings: (400, 768)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.81      0.71      0.76        66
           1       0.74      0.83      0.79        66

    accuracy                           0.77       132
   macro avg       0.78      0.77      0.77       132
weighted avg       0.78      0.77      0.77       132



In [12]:
df_full = pd.read_parquet(DATA_DIR / 'tweets_192k_labeled.parquet')

df_full = df_full.dropna(subset=["clean_text", "unsupervised_sentiment"]).reset_index(drop=True)


In [13]:
import glob
import math

texts_full = df_full["clean_text"].tolist()

save_path = DATA_DIR / 'embeddings_bertweetbr'

batch_size_full = 16
expected_batches = math.ceil(len(texts_full) / batch_size_full)
existing = glob.glob(os.path.join(save_path, "embeddings_*.npy"))

if len(existing) != expected_batches:
    print(f"Embeddings missing or count mismatch ({len(existing)} vs {expected_batches}). Generating...")
    get_embeddings_to_disk(
        texts_full,
        save_path=save_path,
        batch_size=batch_size_full
    )
else:
    print("Embeddings already present. Skipping generation.")


Embeddings already present. Skipping generation.


In [14]:
import glob
import re

files = sorted(
    glob.glob(DATA_DIR / 'embeddings_bertweetbr' / '*.npy'),
    key=lambda p: int(re.search(r"embeddings_(\d+)\.npy", p).group(1))
)

y_hat_all = np.empty(len(df_full), dtype="float32")

cursor = 0

for f in tqdm(files):
    X_batch = np.load(f, mmap_mode="r")
    y_hat_batch = clf.predict_proba(X_batch)[:, 1].astype("float32")
    n = len(y_hat_batch)

    y_hat_all[cursor:cursor + n] = y_hat_batch
    cursor += n

assert cursor == len(df_full), f"Got {cursor} predictions for {len(df_full)} rows"


100%|██████████| 12060/12060 [02:46<00:00, 72.22it/s]


In [20]:
df_full["is_economic_hat_prob"] = y_hat_all
df_full["is_economic_hat"] = (df_full["is_economic_hat_prob"] >= 0.5).astype(int)

In [23]:
df_full["is_economic_hat"].value_counts(normalize=True)

is_economic_hat
0    0.779409
1    0.220591
Name: proportion, dtype: float64

In [ ]:
# Save results with probabilities
out_path = DATA_DIR / 'tweets_192k_labeled_with_probs.parquet'
df_full.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
